# SINet-V2 + PVT-v2 + BGNet — Training on Kaggle

**Pipeline tích hợp BGNet:**  
`PVT-v2 → RFB×3 → EAM (pvt_feats[0]+pvt_feats[3]→edge) → EFM×3 → NCD → RS5 → RS4 → RS3`

**Modules BGNet thêm vào SINet-V2:**
- **EAM** (Edge-Aware Module): pvt_feats[0] (64ch, 88×88, low-level) + pvt_feats[3] (512ch, 11×11, high-level) → edge map
- **EFM** (Edge-guidance Feature Module): inject edge vào từng RFB output qua Local Channel Attention
- **Loss**: L_struct×4 (mask) + λ × L_dice (edge),  λ=3 theo BGNet paper

**Trước khi chạy:**
1. GPU: `Session → Accelerator → GPU T4 x2` hoặc `P100`
2. Internet: `Session → Internet → On`
3. Dataset `sinet-v2-data` phải có tại `/kaggle/input/`

## Bước 1 — Cài thư viện

In [1]:
!pip install thop tensorboardX timm imageio gdown -q
print('✅ Cài xong!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 106.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

## Bước 2 — Kiểm tra GPU

In [2]:
import os, torch
print('=== GPU ===')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('\n=== Input datasets ===')
!ls /kaggle/input/

=== GPU ===
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
CUDA: True
GPU: Tesla T4

=== Input datasets ===
datasets


## Bước 3 — Clone SINet-V2

In [3]:
import os
os.chdir('/kaggle/working')
if not os.path.exists('SINet-V2'):
    !git clone https://github.com/GewelsJI/SINet-V2.git
    print('✅ Clone xong!')
else:
    print('✅ Repo đã có.')
os.chdir('/kaggle/working/SINet-V2')
os.makedirs('./lib', exist_ok=True)
print('Thư mục hiện tại:', os.getcwd())

Cloning into 'SINet-V2'...
remote: Enumerating objects: 509, done.
remote: Counting objects: 100% (244/244), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 509 (delta 178), reused 189 (delta 128), pack-reused 265 (from 1)
Receiving objects: 100% (509/509), 1.82 MiB | 12.82 MiB/s, done.
Resolving deltas: 100% (308/308), done.
✅ Clone xong!
Thư mục hiện tại: /kaggle/working/SINet-V2


## Bước 4 — Ghi `lib/Network_BGNet_PVTv2.py`

**Kiến trúc tích hợp BGNet:**
- **EAM**: pvt_feats[0] (64ch, low-level edge details) + pvt_feats[3] (512ch, high-level location) → edge map `fe`
- **EFM**: `fe ⊗ fi + fi` qua Local Channel Attention → boundary-enhanced features
- Channel mapping PVT-v2-B2: `[64, 128, 320, 512]`

In [4]:
import os
os.chdir('/kaggle/working/SINet-V2')
os.makedirs('./lib', exist_ok=True)

network_code = '# Network_BGNet_PVTv2.py\n# SINet-V2 tich hop BGNet voi PVT-v2 backbone\n#\n# Pipeline: PVT-v2 -> RFB x3 -> EAM -> EFM x3 -> NCD -> RS5 -> RS4 -> RS3\n#\n# BGNet modules:\n#   EAM: pvt_feats[0](64ch, 88x88) + pvt_feats[3](512ch, 11x11) -> edge map fe\n#   EFM: inject edge fe into each RFB output via Local Channel Attention\n#   Loss: structure_loss x4 + lambda_edge * dice_loss(edge)\n#\n# PVT-v2-B2 channel map: [64, 128, 320, 512]\n#   feats[0]: 64ch,  H/4  x W/4   (e.g. 88x88 for 352-input)\n#   feats[1]: 128ch, H/8  x W/8   (e.g. 44x44)\n#   feats[2]: 320ch, H/16 x W/16  (e.g. 22x22)\n#   feats[3]: 512ch, H/32 x W/32  (e.g. 11x11)\n\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n# -----------------------------------------------------------------\n# 1. Backbone loader\n# -----------------------------------------------------------------\ntry:\n    from lib.pvt_v2 import pvt_v2_b2\n    _USE_LOCAL = True\nexcept ImportError:\n    _USE_LOCAL = False\n\nif not _USE_LOCAL:\n    try:\n        import timm\n        _USE_TIMM = True\n    except ImportError:\n        raise ImportError("Cannot find PVT-v2. Run: pip install timm")\n\n\nclass PVTv2_Timm(nn.Module):\n    def __init__(self, pretrained=True):\n        super().__init__()\n        self.model = timm.create_model(\n            "pvt_v2_b2", pretrained=pretrained, features_only=True)\n\n    def forward(self, x):\n        return self.model(x)\n\n\n# -----------------------------------------------------------------\n# 2. Basic building blocks\n# -----------------------------------------------------------------\nclass BasicConv2d(nn.Module):\n    def __init__(self, in_planes, out_planes, kernel_size,\n                 stride=1, padding=0, dilation=1):\n        super().__init__()\n        self.conv = nn.Conv2d(\n            in_planes, out_planes, kernel_size=kernel_size,\n            stride=stride, padding=padding, dilation=dilation, bias=False)\n        self.bn   = nn.BatchNorm2d(out_planes)\n        self.relu = nn.ReLU(inplace=True)\n\n    def forward(self, x):\n        return self.relu(self.bn(self.conv(x)))\n\n\nclass RFB_modified(nn.Module):\n    def __init__(self, in_channel, out_channel):\n        super().__init__()\n        self.relu    = nn.ReLU(True)\n        self.branch0 = nn.Sequential(BasicConv2d(in_channel, out_channel, 1))\n        self.branch1 = nn.Sequential(\n            BasicConv2d(in_channel, out_channel, 1),\n            BasicConv2d(out_channel, out_channel, (1, 3), padding=(0, 1)),\n            BasicConv2d(out_channel, out_channel, (3, 1), padding=(1, 0)),\n            BasicConv2d(out_channel, out_channel, 3, padding=3, dilation=3))\n        self.branch2 = nn.Sequential(\n            BasicConv2d(in_channel, out_channel, 1),\n            BasicConv2d(out_channel, out_channel, (1, 5), padding=(0, 2)),\n            BasicConv2d(out_channel, out_channel, (5, 1), padding=(2, 0)),\n            BasicConv2d(out_channel, out_channel, 3, padding=5, dilation=5))\n        self.branch3 = nn.Sequential(\n            BasicConv2d(in_channel, out_channel, 1),\n            BasicConv2d(out_channel, out_channel, (1, 7), padding=(0, 3)),\n            BasicConv2d(out_channel, out_channel, (7, 1), padding=(3, 0)),\n            BasicConv2d(out_channel, out_channel, 3, padding=7, dilation=7))\n        self.conv_cat = BasicConv2d(4 * out_channel, out_channel, 3, padding=1)\n        self.conv_res = BasicConv2d(in_channel, out_channel, 1)\n\n    def forward(self, x):\n        x0 = self.branch0(x)\n        x1 = self.branch1(x)\n        x2 = self.branch2(x)\n        x3 = self.branch3(x)\n        return self.relu(\n            self.conv_cat(torch.cat((x0, x1, x2, x3), 1)) + self.conv_res(x))\n\n\n# -----------------------------------------------------------------\n# 3. BGNet modules: EAM + EFM\n# -----------------------------------------------------------------\nclass EAM(nn.Module):\n    def __init__(self, c_low=64, c_high=512, c_mid=64, c_high_mid=256):\n        super().__init__()\n        self.conv_low  = BasicConv2d(c_low,  c_mid,      kernel_size=1)\n        self.conv_high = BasicConv2d(c_high, c_high_mid, kernel_size=1)\n        in_cat = c_mid + c_high_mid\n        self.conv_fuse = nn.Sequential(\n            BasicConv2d(in_cat, in_cat, 3, padding=1),\n            BasicConv2d(in_cat, in_cat, 3, padding=1),\n            nn.Conv2d(in_cat, 1, 1),\n            nn.Sigmoid())\n\n    def forward(self, f_low, f_high):\n        fl = self.conv_low(f_low)\n        fh = self.conv_high(f_high)\n        fh = F.interpolate(fh, size=fl.shape[2:], mode="bilinear", align_corners=True)\n        fe = self.conv_fuse(torch.cat([fl, fh], dim=1))\n        return fe\n\n\nclass LocalChannelAttention(nn.Module):\n    def __init__(self, channels):\n        super().__init__()\n        t = int(abs((1 + math.log2(channels)) / 2))\n        k = t if t % 2 == 1 else t + 1\n        self.avg_pool = nn.AdaptiveAvgPool2d(1)\n        self.conv1d   = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)\n        self.sigmoid  = nn.Sigmoid()\n\n    def forward(self, x):\n        y = self.avg_pool(x)\n        y = self.conv1d(y.squeeze(-1).transpose(-1, -2))\n        y = y.transpose(-1, -2).unsqueeze(-1)\n        return x * self.sigmoid(y)\n\n\nclass EFM(nn.Module):\n    def __init__(self, channel=32):\n        super().__init__()\n        self.conv_fuse = BasicConv2d(channel, channel, 3, padding=1)\n        self.lca       = LocalChannelAttention(channel)\n        self.conv_out  = nn.Conv2d(channel, channel, 1)\n\n    def forward(self, fi, fe):\n        fe_a = F.interpolate(fe, size=fi.shape[2:], mode="bilinear", align_corners=True)\n        fie  = self.conv_fuse(fi * fe_a + fi)\n        fia  = self.lca(fie)\n        return self.conv_out(fia * fie)\n\n\n# -----------------------------------------------------------------\n# 4. SINet-V2 decoder\n# -----------------------------------------------------------------\nclass NeighborConnectionDecoder(nn.Module):\n    def __init__(self, channel):\n        super().__init__()\n        self.upsample       = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)\n        self.conv_upsample1 = BasicConv2d(channel, channel, 3, padding=1)\n        self.conv_upsample2 = BasicConv2d(channel, channel, 3, padding=1)\n        self.conv_upsample3 = BasicConv2d(channel, channel, 3, padding=1)\n        self.conv_upsample4 = BasicConv2d(channel, channel, 3, padding=1)\n        self.conv_upsample5 = BasicConv2d(2*channel, 2*channel, 3, padding=1)\n        self.conv_concat2   = BasicConv2d(2*channel, 2*channel, 3, padding=1)\n        self.conv_concat3   = BasicConv2d(3*channel, 3*channel, 3, padding=1)\n        self.conv4          = BasicConv2d(3*channel, 3*channel, 3, padding=1)\n        self.conv5          = nn.Conv2d(3*channel, 1, 1)\n\n    def forward(self, x1, x2, x3):\n        x1_1 = x1\n        x2_1 = self.conv_upsample1(self.upsample(x1)) * x2\n        x3_1 = (self.conv_upsample2(self.upsample(x2_1))\n                * self.conv_upsample3(self.upsample(x2)) * x3)\n        x2_2 = torch.cat((x2_1, self.conv_upsample4(self.upsample(x1_1))), 1)\n        x2_2 = self.conv_concat2(x2_2)\n        x3_2 = torch.cat((x3_1, self.conv_upsample5(self.upsample(x2_2))), 1)\n        x3_2 = self.conv_concat3(x3_2)\n        return self.conv5(self.conv4(x3_2))\n\n\nclass GRA(nn.Module):\n    def __init__(self, channel, subchannel):\n        super().__init__()\n        self.group = channel // subchannel\n        self.conv  = nn.Sequential(\n            nn.Conv2d(channel + self.group, channel, 3, padding=1),\n            nn.ReLU(True))\n        self.score = nn.Conv2d(channel, 1, 3, padding=1)\n\n    def forward(self, x, y):\n        g = self.group\n        if g == 1:\n            x_cat = torch.cat((x, y), 1)\n        else:\n            xs    = torch.chunk(x, g, dim=1)\n            parts = []\n            for xi in xs:\n                parts += [xi, y]\n            x_cat = torch.cat(parts, 1)\n        x = x + self.conv(x_cat)\n        y = y + self.score(x)\n        return x, y\n\n\nclass ReverseStage(nn.Module):\n    def __init__(self, channel):\n        super().__init__()\n        self.weak_gra   = GRA(channel, channel)\n        self.medium_gra = GRA(channel, 8)\n        self.strong_gra = GRA(channel, 1)\n\n    def forward(self, x, y):\n        y = -1 * torch.sigmoid(y) + 1\n        x, y = self.weak_gra(x, y)\n        x, y = self.medium_gra(x, y)\n        _, y = self.strong_gra(x, y)\n        return y\n\n\n# -----------------------------------------------------------------\n# 5. Unified Network\n# -----------------------------------------------------------------\nPVT_CHANNELS = {\n    "b0": [32,  64,  160, 256],\n    "b1": [64,  128, 320, 512],\n    "b2": [64,  128, 320, 512],\n    "b3": [64,  128, 320, 512],\n    "b4": [64,  128, 320, 512],\n    "b5": [64,  128, 320, 512],\n}\n\n\nclass Network(nn.Module):\n    def __init__(self, channel=32, pvt_variant="b2", imagenet_pretrained=True):\n        super().__init__()\n        if _USE_LOCAL:\n            fn_map = {"b2": pvt_v2_b2}\n            self.backbone = fn_map[pvt_variant](pretrained=imagenet_pretrained)\n        else:\n            self.backbone = PVTv2_Timm(pretrained=imagenet_pretrained)\n\n        ch = PVT_CHANNELS[pvt_variant]\n\n        self.rfb2_1 = RFB_modified(ch[1], channel)\n        self.rfb3_1 = RFB_modified(ch[2], channel)\n        self.rfb4_1 = RFB_modified(ch[3], channel)\n\n        self.eam = EAM(c_low=ch[0], c_high=ch[3], c_mid=64, c_high_mid=256)\n\n        self.efm2 = EFM(channel)\n        self.efm3 = EFM(channel)\n        self.efm4 = EFM(channel)\n\n        self.NCD = NeighborConnectionDecoder(channel)\n        self.RS5 = ReverseStage(channel)\n        self.RS4 = ReverseStage(channel)\n        self.RS3 = ReverseStage(channel)\n\n    def forward(self, x):\n        feats = self.backbone(x)\n\n        x2_rfb = self.rfb2_1(feats[1])\n        x3_rfb = self.rfb3_1(feats[2])\n        x4_rfb = self.rfb4_1(feats[3])\n\n        fe = self.eam(feats[0], feats[3])\n        edge_pred = F.interpolate(fe, scale_factor=4, mode="bilinear", align_corners=False)\n\n        x2_efm = self.efm2(x2_rfb, fe)\n        x3_efm = self.efm3(x3_rfb, fe)\n        x4_efm = self.efm4(x4_rfb, fe)\n\n        S_g      = self.NCD(x4_efm, x3_efm, x2_efm)\n        S_g_pred = F.interpolate(S_g, scale_factor=8, mode="bilinear", align_corners=False)\n\n        g5  = F.interpolate(S_g, scale_factor=0.25, mode="bilinear", align_corners=False)\n        ra4 = self.RS5(x4_efm, g5)\n        S5  = ra4 + g5\n        S5_pred = F.interpolate(S5, scale_factor=32, mode="bilinear", align_corners=False)\n\n        g4  = F.interpolate(S5, scale_factor=2, mode="bilinear", align_corners=False)\n        ra3 = self.RS4(x3_efm, g4)\n        S4  = ra3 + g4\n        S4_pred = F.interpolate(S4, scale_factor=16, mode="bilinear", align_corners=False)\n\n        g3  = F.interpolate(S4, scale_factor=2, mode="bilinear", align_corners=False)\n        ra2 = self.RS3(x2_efm, g3)\n        S3  = ra2 + g3\n        S3_pred = F.interpolate(S3, scale_factor=8, mode="bilinear", align_corners=False)\n\n        return S_g_pred, S5_pred, S4_pred, S3_pred, edge_pred\n'

with open('./lib/Network_BGNet_PVTv2.py', 'w') as f:
    f.write(network_code)

sz = os.path.getsize('./lib/Network_BGNet_PVTv2.py')
print(f'✅ Ghi xong lib/Network_BGNet_PVTv2.py  ({sz:,} bytes)')

✅ Ghi xong lib/Network_BGNet_PVTv2.py  (10,942 bytes)


## Bước 5 — Ghi `MyTrain_Val_BGNet.py`

**Loss tích hợp BGNet:**  
`L_total = L_struct(S_g) + L_struct(S5) + L_struct(S4) + L_struct(S3) + λ × L_dice(edge)`  
- λ = 3 (theo BGNet paper)
- `L_struct = W-BCE + W-IoU`  (SINet-V2)
- `L_dice` xử lý class imbalance cho edge supervision  
- Edge GT được tạo từ binary mask bằng morphological boundary approximation

In [5]:
import os
os.chdir('/kaggle/working/SINet-V2')

train_code = '# MyTrain_Val_BGNet.py\nimport os, csv\nimport torch\nimport torch.nn.functional as F\nimport numpy as np\nfrom datetime import datetime\nfrom lib.Network_BGNet_PVTv2 import Network\nfrom utils.data_val import get_loader, test_dataset\nfrom utils.utils import clip_gradient\nfrom tensorboardX import SummaryWriter\nimport logging\nimport torch.backends.cudnn as cudnn\n\nbest_mae   = 1.0\nbest_epoch = 0\nstep       = 0\n\n\ndef structure_loss(pred, mask):\n    weit  = 1 + 5 * torch.abs(\n        F.avg_pool2d(mask, kernel_size=31, stride=1, padding=15) - mask)\n    wbce  = F.binary_cross_entropy_with_logits(pred, mask, reduction="none")\n    wbce  = (weit * wbce).sum(dim=(2, 3)) / weit.sum(dim=(2, 3))\n    pred  = torch.sigmoid(pred)\n    inter = ((pred * mask) * weit).sum(dim=(2, 3))\n    union = ((pred + mask) * weit).sum(dim=(2, 3))\n    wiou  = 1 - (inter + 1) / (union - inter + 1)\n    return (wbce + wiou).mean()\n\n\ndef dice_loss(pred, mask):\n    smooth = 1.0\n    p = pred.view(pred.size(0), -1)\n    m = mask.view(mask.size(0), -1)\n    inter = (p * m).sum(dim=1)\n    union = p.sum(dim=1) + m.sum(dim=1)\n    return (1 - (2 * inter + smooth) / (union + smooth)).mean()\n\n\ndef get_edge_gt(mask):\n    kernel = 5\n    pad    = kernel // 2\n    dilate = F.max_pool2d(mask, kernel_size=kernel, stride=1, padding=pad)\n    erode  = -F.max_pool2d(-mask, kernel_size=kernel, stride=1, padding=pad)\n    return (dilate - erode).clamp(0, 1)\n\n\ndef cosine_lr_with_warmup(optimizer, base_lr, epoch, total_epoch,\n                          warmup_epoch=3, lr_min=1e-6):\n    if epoch <= warmup_epoch:\n        lr = base_lr * 0.1 + (base_lr - base_lr * 0.1) * (epoch / warmup_epoch)\n    else:\n        progress = (epoch - warmup_epoch) / max(total_epoch - warmup_epoch, 1)\n        lr = lr_min + 0.5 * (base_lr - lr_min) * (1 + np.cos(np.pi * progress))\n    for pg in optimizer.param_groups:\n        pg["lr"] = lr * 0.1 if pg.get("is_backbone", False) else lr\n    return lr\n\n\ndef init_csv(path):\n    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)\n    with open(path, "w", newline="") as f:\n        csv.writer(f).writerow([\n            "epoch", "lr", "loss_total", "loss_mask", "loss_edge",\n            "mae", "best_mae", "best_epoch"])\n\n\ndef log_csv(path, epoch, lr, lt, lm, le, mae, bm, be):\n    with open(path, "a", newline="") as f:\n        csv.writer(f).writerow([\n            epoch, f"{lr:.2e}", f"{lt:.4f}", f"{lm:.4f}",\n            f"{le:.4f}", f"{mae:.6f}", f"{bm:.6f}", be])\n\n\ndef train(train_loader, model, optimizer, epoch, save_path, writer, opt):\n    global step\n    model.train()\n    loss_all = mask_all = edge_all = 0\n    n = 0\n    try:\n        for i, (images, gts) in enumerate(train_loader, start=1):\n            optimizer.zero_grad()\n            images, gts = images.cuda(), gts.cuda()\n\n            S_g, S5, S4, S3, edge_pred = model(images)\n\n            loss_mask = (structure_loss(S_g, gts)\n                         + structure_loss(S5, gts)\n                         + structure_loss(S4, gts)\n                         + structure_loss(S3, gts))\n\n            edge_gt   = get_edge_gt(gts)\n            loss_edge = dice_loss(edge_pred, edge_gt)\n\n            loss = loss_mask + opt.lambda_edge * loss_edge\n            loss.backward()\n            clip_gradient(optimizer, opt.clip)\n            optimizer.step()\n\n            step += 1; n += 1\n            loss_all += loss.item()\n            mask_all += loss_mask.item()\n            edge_all += loss_edge.item()\n\n            if i % 50 == 0 or i == total_step or i == 1:\n                msg = (f"{datetime.now()} "\n                       f"Ep[{epoch:03d}/{opt.epoch}] "\n                       f"Step[{i:04d}/{total_step}] "\n                       f"Total:{loss.item():.4f} "\n                       f"Mask:{loss_mask.item():.4f} "\n                       f"Edge:{loss_edge.item():.4f}")\n                print(msg)\n                logging.info(msg)\n                writer.add_scalars("Loss", {\n                    "mask":  loss_mask.item(),\n                    "edge":  loss_edge.item(),\n                    "total": loss.item()}, global_step=step)\n\n        avg_lt = loss_all / n\n        avg_lm = mask_all / n\n        avg_le = edge_all / n\n        logging.info(f"[Train] Ep[{epoch}/{opt.epoch}] "\n                     f"Avg Loss:{avg_lt:.4f} Mask:{avg_lm:.4f} Edge:{avg_le:.4f}")\n        writer.add_scalar("Loss-epoch/total", avg_lt, global_step=epoch)\n        writer.add_scalar("Loss-epoch/mask",  avg_lm, global_step=epoch)\n        writer.add_scalar("Loss-epoch/edge",  avg_le, global_step=epoch)\n\n        if epoch % 5 == 0:\n            p = save_path + f"Net_epoch_{epoch}.pth"\n            torch.save(model.state_dict(), p)\n            print(f"[Saved] {p}")\n        return avg_lt, avg_lm, avg_le\n\n    except KeyboardInterrupt:\n        torch.save(model.state_dict(),\n                   save_path + f"Net_epoch_{epoch}_interrupt.pth")\n        raise\n\n\ndef val(test_loader, model, epoch, save_path, writer):\n    global best_mae, best_epoch\n    model.eval()\n    mae_sum = 0\n    with torch.no_grad():\n        for _ in range(test_loader.size):\n            image, gt, name, _ = test_loader.load_data()\n            gt = np.asarray(gt, np.float32)\n            gt /= (gt.max() + 1e-8)\n            res = model(image.cuda())[3]\n            res = F.interpolate(res, size=gt.shape,\n                                mode="bilinear", align_corners=False)\n            res = res.sigmoid().cpu().numpy().squeeze()\n            res = (res - res.min()) / (res.max() - res.min() + 1e-8)\n            mae_sum += np.abs(res - gt).sum() / (gt.shape[0] * gt.shape[1])\n\n    mae = mae_sum / test_loader.size\n    writer.add_scalar("MAE", torch.tensor(mae), global_step=epoch)\n    print(f"Ep:{epoch}  MAE:{mae:.6f}  "\n          f"bestMAE:{best_mae:.6f}  bestEp:{best_epoch}")\n\n    if epoch == 1 or mae < best_mae:\n        best_mae = mae; best_epoch = epoch\n        p = save_path + "Net_epoch_best.pth"\n        torch.save(model.state_dict(), p)\n        print(f"  -> New best! Saved epoch {epoch}")\n    logging.info(f"[Val] Ep:{epoch} MAE:{mae:.6f} "\n                 f"best:{best_mae:.6f} ep:{best_epoch}")\n    return mae\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--epoch",        type=int,   default=25)\n    parser.add_argument("--lr",           type=float, default=5e-4)\n    parser.add_argument("--batchsize",    type=int,   default=32)\n    parser.add_argument("--trainsize",    type=int,   default=352)\n    parser.add_argument("--clip",         type=float, default=0.5)\n    parser.add_argument("--warmup_epoch", type=int,   default=3)\n    parser.add_argument("--lr_min",       type=float, default=1e-6)\n    parser.add_argument("--lambda_edge",  type=float, default=3.0)\n    parser.add_argument("--load",         type=str,   default=None)\n    parser.add_argument("--gpu_id",       type=str,   default="0")\n    parser.add_argument("--pvt_variant",  type=str,   default="b2")\n    parser.add_argument("--train_root",   type=str,\n                        default="./Dataset/TrainValDataset/")\n    parser.add_argument("--val_root",     type=str,\n                        default="./Dataset/TestDataset/CAMO/")\n    parser.add_argument("--save_path",    type=str,\n                        default="./snapshot/SINet_V2_BGNet_PVTv2/")\n    opt = parser.parse_args()\n\n    os.environ["CUDA_VISIBLE_DEVICES"] = opt.gpu_id\n    cudnn.benchmark = True\n\n    os.makedirs(opt.save_path, exist_ok=True)\n    logging.basicConfig(\n        filename=opt.save_path + "log.log",\n        format="[%(asctime)s-%(filename)s-%(levelname)s:%(message)s]",\n        level=logging.INFO, filemode="a", datefmt="%Y-%m-%d %I:%M:%S %p")\n    logging.info(\n        f">>> Config: epoch={opt.epoch} lr={opt.lr:.2e} "\n        f"batch={opt.batchsize} size={opt.trainsize} "\n        f"lambda_edge={opt.lambda_edge}")\n\n    CSV_LOG = opt.save_path + "training_log.csv"\n    init_csv(CSV_LOG)\n\n    model = Network(channel=32, pvt_variant=opt.pvt_variant,\n                    imagenet_pretrained=True).cuda()\n    if opt.load:\n        model.load_state_dict(torch.load(opt.load))\n        print(f"Loaded: {opt.load}")\n\n    backbone_params = list(model.backbone.parameters())\n    head_params     = [p for p in model.parameters()\n                       if not any(p is q for q in backbone_params)]\n    optimizer = torch.optim.Adam([\n        {"params": backbone_params, "lr": opt.lr * 0.1, "is_backbone": True},\n        {"params": head_params,     "lr": opt.lr}], lr=opt.lr)\n\n    train_loader = get_loader(\n        opt.train_root + "Imgs/", opt.train_root + "GT/",\n        batchsize=opt.batchsize, trainsize=opt.trainsize, num_workers=4)\n    test_loader  = test_dataset(\n        opt.val_root + "Imgs/", opt.val_root + "GT/", opt.trainsize)\n    total_step   = len(train_loader)\n\n    writer = SummaryWriter(opt.save_path + "summary")\n    print(f"Training steps/epoch: {total_step}")\n    print(f"Lambda edge: {opt.lambda_edge}")\n\n    for epoch in range(1, opt.epoch + 1):\n        cur_lr = cosine_lr_with_warmup(\n            optimizer, opt.lr, epoch, opt.epoch,\n            opt.warmup_epoch, opt.lr_min)\n        lt, lm, le = train(\n            train_loader, model, optimizer, epoch, opt.save_path, writer, opt)\n        mae = val(test_loader, model, epoch, opt.save_path, writer)\n        log_csv(CSV_LOG, epoch, cur_lr, lt, lm, le, mae, best_mae, best_epoch)\n        print(f"[Ep {epoch:3d}/{opt.epoch}] LR={cur_lr:.2e} "\n              f"Loss={lt:.4f} Mask={lm:.4f} Edge={le:.4f} MAE={mae:.6f}")\n\n    writer.close()\n    print("Training done!")\n'

with open('./MyTrain_Val_BGNet.py', 'w') as f:
    f.write(train_code)

sz = os.path.getsize('./MyTrain_Val_BGNet.py')
print(f'✅ Ghi xong MyTrain_Val_BGNet.py  ({sz:,} bytes)')

✅ Ghi xong MyTrain_Val_BGNet.py  (9,545 bytes)


## Bước 6 — Chuẩn bị Dataset

Dataset cần cấu trúc:
```
Dataset/
  TrainValDataset/
    Imgs/   (ảnh train)
    GT/     (mask GT train)
  TestDataset/
    CAMO/   Imgs/ + GT/
    COD10K/ Imgs/ + GT/
    CHAMELEON/ Imgs/ + GT/
```

In [6]:
import shutil, os
os.chdir('/kaggle/working/SINet-V2')

INPUT_BASE = '/kaggle/input/datasets/tanleluv/sinet-v2-dataset/sinet_v2_dataset'
TRAIN_SRC  = f'{INPUT_BASE}/COD-TrainDataset'
TEST_SRC   = f'{INPUT_BASE}/COD-TestDataset'

TRAIN_DIR  = './Dataset/TrainValDataset'
TEST_DIR   = './Dataset/TestDataset'

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR,  exist_ok=True)

if not os.path.exists(f'{TRAIN_DIR}/Imgs'):
    print('Copying train dataset...')
    os.system(f'cp -r {TRAIN_SRC}/. {TRAIN_DIR}/')
    print('✅ Train dataset ready!')
else:
    print('✅ Train dataset already exists.')

if not os.path.exists(f'{TEST_DIR}/CAMO'):
    print('Copying test dataset...')
    os.system(f'cp -r {TEST_SRC}/. {TEST_DIR}/')
    print('✅ Test dataset ready!')
else:
    print('✅ Test dataset already exists.')

print('\n=== Dataset structure ===')
for root, dirs, files in os.walk('./Dataset'):
    depth = root.replace('./Dataset', '').count(os.sep)
    if depth > 2: continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth == 2:
        print(f'{indent}  ({len(files)} files)')

Copying train dataset...
✅ Train dataset ready!
Copying test dataset...
✅ Test dataset ready!

=== Dataset structure ===
Dataset/
  TrainValDataset/
    Imgs/
      (4040 files)
    GT/
      (4040 files)
  TestDataset/
    COD10K/
      (0 files)
    CAMO/
      (0 files)
    NC4K/
      (0 files)
    CHAMELEON/
      (0 files)


## Bước 7 — Kiểm tra Model BGNet (sanity check)

In [7]:
import sys, torch, torch.nn.functional as F
sys.path.insert(0, '/kaggle/working/SINet-V2')
import os; os.chdir('/kaggle/working/SINet-V2')

from lib.Network_BGNet_PVTv2 import Network

model = Network(channel=32, pvt_variant='b2', imagenet_pretrained=True).cuda()
dummy = torch.randn(2, 3, 352, 352).cuda()
with torch.no_grad():
    outs = model(dummy)

labels = ['S_g_pred (coarse NCD)', 'S5_pred (RS5)',
          'S4_pred (RS4)', 'S3_pred (RS3 final)', 'edge_pred (EAM)']
print('Output shapes:')
for i, (o, lbl) in enumerate(zip(outs, labels)):
    print(f'  outs[{i}] {lbl:30s}: {tuple(o.shape)}')

total  = sum(p.numel() for p in model.parameters())
bb     = sum(p.numel() for p in model.backbone.parameters())
head   = total - bb
print(f'\nTotal params   : {total/1e6:.2f}M')
print(f'Backbone params: {bb/1e6:.2f}M  (PVT-v2-B2)')
print(f'Head params    : {head/1e6:.2f}M  (RFB+EAM+EFM+NCD+RS)')

edge = outs[4]
print(f'\nEdge range: [{edge.min().item():.4f}, {edge.max().item():.4f}]  (expect [0,1])')

del model, dummy
torch.cuda.empty_cache()
print('✅ Model BGNet OK!')

model.safetensors:   0%|          | 0.00/101M [00:00<?, ?B/s]

Output shapes:
  outs[0] S_g_pred (coarse NCD)         : (2, 1, 352, 352)
  outs[1] S5_pred (RS5)                 : (2, 1, 352, 352)
  outs[2] S4_pred (RS4)                 : (2, 1, 352, 352)
  outs[3] S3_pred (RS3 final)           : (2, 1, 352, 352)
  outs[4] edge_pred (EAM)               : (2, 1, 352, 352)

Total params   : 27.70M
Backbone params: 24.85M  (PVT-v2-B2)
Head params    : 2.85M  (RFB+EAM+EFM+NCD+RS)

Edge range: [0.1817, 0.7566]  (expect [0,1])
✅ Model BGNet OK!


## Bước 8 — Train 50 Epoch

> Checkpoint lưu mỗi 5 epoch tại `snapshot/SINet_V2_BGNet_PVTv2/`  
> Log CSV: `snapshot/SINet_V2_BGNet_PVTv2/training_log.csv`

In [8]:
import time, os
os.chdir('/kaggle/working/SINet-V2')
os.makedirs('./snapshot/SINet_V2_BGNet_PVTv2', exist_ok=True)

print('Training SINet-V2 + BGNet + PVT-v2...')
start = time.time()

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python MyTrain_Val_BGNet.py \
    --epoch 50 \
    --lr 5e-4 \
    --batchsize 8 \
    --trainsize 352 \
    --clip 0.5 \
    --warmup_epoch 3 \
    --lr_min 1e-6 \
    --lambda_edge 3.0 \
    --pvt_variant b2 \
    --gpu_id 0 \
    --save_path ./snapshot/SINet_V2_BGNet_PVTv2/ \
    --train_root ./Dataset/TrainValDataset/ \
    --val_root   ./Dataset/TestDataset/CAMO/

elapsed = (time.time() - start) / 3600
print(f'Total time: {elapsed:.2f} hours')

Training SINet-V2 + BGNet + PVT-v2...
Training steps/epoch: 505
Lambda edge: 3.0
2026-06-09 04:59:45.683190 Ep[001/50] Step[0001/505] Total:9.9951 Mask:7.2395 Edge:0.9185
2026-06-09 05:00:16.779987 Ep[001/50] Step[0050/505] Total:6.5704 Mask:4.2616 Edge:0.7696
2026-06-09 05:00:49.982898 Ep[001/50] Step[0100/505] Total:6.6345 Mask:4.2438 Edge:0.7969
2026-06-09 05:01:24.600629 Ep[001/50] Step[0150/505] Total:6.5240 Mask:4.2535 Edge:0.7568
2026-06-09 05:02:01.091888 Ep[001/50] Step[0200/505] Total:5.7405 Mask:3.3756 Edge:0.7883
2026-06-09 05:02:38.608700 Ep[001/50] Step[0250/505] Total:5.7045 Mask:3.5703 Edge:0.7114
2026-06-09 05:03:15.705904 Ep[001/50] Step[0300/505] Total:5.9025 Mask:3.4840 Edge:0.8062
2026-06-09 05:03:52.760383 Ep[001/50] Step[0350/505] Total:5.1127 Mask:2.9438 Edge:0.7230
2026-06-09 05:04:30.099817 Ep[001/50] Step[0400/505] Total:5.7329 Mask:3.5208 Edge:0.7374
2026-06-09 05:05:07.129189 Ep[001/50] Step[0450/505] Total:5.1369 Mask:2.9875 Edge:0.7164
2026-06-09 05:05:44

## Bước 9 — Vẽ biểu đồ Loss và MAE

In [9]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np, os

CSV_PATH = '/kaggle/working/SINet-V2/snapshot/SINet_V2_BGNet_PVTv2/training_log.csv'
PLOT_DIR = '/kaggle/working/plots_bgnet/'
os.makedirs(PLOT_DIR, exist_ok=True)

assert os.path.exists(CSV_PATH), f'Chua co log: {CSV_PATH}'
df = pd.read_csv(CSV_PATH)
for col in ['epoch', 'best_epoch']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
for col in ['lr', 'loss_total', 'loss_mask', 'loss_edge', 'mae', 'best_mae']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Total epochs trained: {len(df)}')
print(df.to_string(index=False))

best_row   = df.loc[df['mae'].idxmin()]
best_ep    = int(best_row['epoch'])
best_mae_v = float(best_row['mae'])
print(f'\nBest MAE = {best_mae_v:.6f} at Epoch {best_ep}')
epochs = df['epoch'].values

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 12,
    'axes.titlesize': 14, 'axes.labelsize': 12,
    'legend.fontsize': 11, 'lines.linewidth': 2,
    'axes.grid': True, 'grid.alpha': 0.35,
})

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, df['loss_total'], color='#E63946', label='Total loss',   lw=2.5)
ax.plot(epochs, df['loss_mask'],  color='#457B9D', label='Mask loss',    lw=1.8, ls='--')
ax.plot(epochs, df['loss_edge'],  color='#F4A261', label='Edge loss x3', lw=1.8, ls=':')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('SINet-V2 + BGNet (PVT-v2) — Training Loss')
ax.legend(loc='upper right')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
p1 = PLOT_DIR + 'loss_curve.png'
plt.savefig(p1, dpi=150, bbox_inches='tight'); plt.close()
print(f'Saved: {p1}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, df['mae'],      color='#E76F51', label='Val MAE (CAMO)', lw=2.5)
ax.plot(epochs, df['best_mae'], color='#2A9D8F', label='Best MAE so far',lw=1.5, ls='--', alpha=0.8)
ax.scatter([best_ep], [best_mae_v], color='red', zorder=5, s=80)
ax.annotate(
    f'Best Ep {best_ep}\n{best_mae_v:.4f}',
    xy=(best_ep, best_mae_v),
    xytext=(best_ep + max(1, len(df)//10), best_mae_v + 0.003),
    fontsize=10, color='red',
    arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
ax.set_xlabel('Epoch'); ax.set_ylabel('MAE')
ax.set_title('SINet-V2 + BGNet (PVT-v2) — Validation MAE (CAMO)')
ax.legend(loc='upper right')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
p2 = PLOT_DIR + 'mae_curve.png'
plt.savefig(p2, dpi=150, bbox_inches='tight'); plt.close()
print(f'Saved: {p2}')

fig, ax1 = plt.subplots(figsize=(12, 5))
c_loss = '#E63946'; c_mae = '#457B9D'
ax1.plot(epochs, df['loss_total'], color=c_loss, lw=2.5, label='Total loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss', color=c_loss)
ax1.tick_params(axis='y', labelcolor=c_loss)
ax2 = ax1.twinx()
ax2.plot(epochs, df['mae'], color=c_mae, lw=2.5, ls='--', label='Val MAE')
ax2.set_ylabel('MAE', color=c_mae)
ax2.tick_params(axis='y', labelcolor=c_mae)
ax2.scatter([best_ep], [best_mae_v], color='red', zorder=5, s=80)
lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lbl1 + lbl2, loc='upper right')
ax1.set_title('SINet-V2 + BGNet — Loss & MAE')
ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax1.grid(True, alpha=0.3)
plt.tight_layout()
p3 = PLOT_DIR + 'loss_mae_combined.png'
plt.savefig(p3, dpi=150, bbox_inches='tight'); plt.close()
print(f'Saved: {p3}')

print(f'\n✅ All plots saved to {PLOT_DIR}')

Total epochs trained: 50
 epoch       lr  loss_total  loss_mask  loss_edge      mae  best_mae  best_epoch
     1 0.000200      5.8811     3.6167     0.7548 0.097878  0.097878           1
     2 0.000350      4.8160     2.8631     0.6510 0.079682  0.079682           2
     3 0.000500      4.4702     2.6251     0.6150 0.080660  0.079682           2
     4 0.000499      4.2056     2.4355     0.5900 0.074669  0.074669           4
     5 0.000498      4.0142     2.2954     0.5730 0.074051  0.074051           5
     6 0.000495      3.8837     2.2046     0.5597 0.066789  0.066789           6
     7 0.000491      3.8239     2.1660     0.5526 0.064166  0.064166           7
     8 0.000486      3.6788     2.0604     0.5394 0.060533  0.060533           8
     9 0.000480      3.6486     2.0434     0.5351 0.062988  0.060533           8
    10 0.000473      3.5634     1.9867     0.5256 0.064623  0.060533           8
    11 0.000465      3.4949     1.9366     0.5194 0.061778  0.060533           8
   

## Bước 10 — Inference trên 3 bộ test (CAMO / COD10K / CHAMELEON)

In [10]:
import imageio, torch, torch.nn.functional as F
import numpy as np, os, sys

WORK = '/kaggle/working/SINet-V2'
sys.path.insert(0, WORK)
os.chdir(WORK)

from lib.Network_BGNet_PVTv2 import Network
from utils.data_val import test_dataset

MODEL_PATH = './snapshot/SINet_V2_BGNet_PVTv2/Net_epoch_best.pth'
assert os.path.exists(MODEL_PATH), f'Not found: {MODEL_PATH}'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Model: {MODEL_PATH}  Device: {device}')

mae_results = {}
for dataset in ['CAMO', 'COD10K', 'CHAMELEON']:
    img_dir = f'./Dataset/TestDataset/{dataset}/Imgs/'
    gt_dir  = f'./Dataset/TestDataset/{dataset}/GT/'
    if not os.path.exists(img_dir):
        print(f'Skip {dataset} (no data)')
        continue
    save_path = f'./res/SINet_V2_BGNet_PVTv2/{dataset}/'
    os.makedirs(save_path, exist_ok=True)

    model = Network(channel=32, pvt_variant='b2', imagenet_pretrained=False)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.to(device).eval()

    loader  = test_dataset(img_dir, gt_dir, 352)
    mae_sum = 0
    for _ in range(loader.size):
        image, gt, name, _ = loader.load_data()
        gt = np.asarray(gt, np.float32)
        gt /= (gt.max() + 1e-8)
        with torch.no_grad():
            res = model(image.to(device))[3]
        res = F.interpolate(res, size=gt.shape, mode='bilinear', align_corners=False)
        res = res.sigmoid().cpu().numpy().squeeze()
        res = (res - res.min()) / (res.max() - res.min() + 1e-8)
        mae_sum += np.abs(res - gt).sum() / (gt.shape[0] * gt.shape[1])
        imageio.imwrite(save_path + name, (res * 255).astype(np.uint8))

    mae_results[dataset] = mae_sum / loader.size
    del model
    torch.cuda.empty_cache()
    print(f'{dataset:10s}  {loader.size} images  MAE = {mae_results[dataset]:.6f}')

print('\n=== Final MAE ===')
for ds, mae in mae_results.items():
    print(f'  {ds:10s}: {mae:.6f}')

Model: ./snapshot/SINet_V2_BGNet_PVTv2/Net_epoch_best.pth  Device: cuda
CAMO        250 images  MAE = 0.050988
COD10K      2026 images  MAE = 0.028665
CHAMELEON   76 images  MAE = 0.027427

=== Final MAE ===
  CAMO      : 0.050988
  COD10K    : 0.028665
  CHAMELEON : 0.027427


## Bước 11 — Lưu Training Summary (JSON)

In [11]:
import json, pandas as pd, numpy as np, os

CSV_PATH = '/kaggle/working/SINet-V2/snapshot/SINet_V2_BGNet_PVTv2/training_log.csv'
df = pd.read_csv(CSV_PATH)
for col in ['epoch', 'best_epoch']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
for col in ['lr', 'loss_total', 'loss_mask', 'loss_edge', 'mae', 'best_mae']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

best_row = df.loc[df['mae'].idxmin()]

summary = {
    'model': 'SINet-V2 + BGNet + PVT-v2-B2',
    'architecture': {
        'backbone':  'PVT-v2-B2 (pretrained ImageNet)',
        'neck':      'RFB_modified x3 (128/320/512 -> 32ch)',
        'bgnet_eam': 'EAM: pvt_feats[0](64ch) + pvt_feats[3](512ch) -> edge',
        'bgnet_efm': 'EFM x3: edge injection via LocalChannelAttention',
        'decoder':   'NCD -> RS5 -> RS4 -> RS3',
        'outputs':   '4 mask predictions + 1 edge prediction',
    },
    'hyperparameters': {
        'total_epochs':  70,
        'lr_head':       5e-4,
        'lr_backbone':   5e-5,
        'lr_schedule':   'cosine annealing + 3-epoch warmup',
        'batch_size':    32,
        'input_size':    352,
        'optimizer':     'Adam',
        'clip_gradient': 0.5,
        'lambda_edge':   3.0,
        'loss_mask':     'Weighted BCE + Weighted IoU',
        'loss_edge':     'Dice loss (BGNet paper)',
    },
    'datasets': {
        'train': 'COD10K-train + CAMO-train (4040 images)',
        'val':   'CAMO test set',
    },
    'results': {
        'best_epoch':    int(best_row['epoch']),
        'best_mae_camo': float(best_row['mae']),
        'final_loss':    float(df.iloc[-1]['loss_total']),
        'mae_per_dataset': ({k: float(v) for k, v in mae_results.items()}
                            if 'mae_results' in dir() else {}),
    },
}

out = '/kaggle/working/training_summary_bgnet.json'
with open(out, 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print('Saved:', out)
print(json.dumps(summary, indent=2, ensure_ascii=False))

Saved: /kaggle/working/training_summary_bgnet.json
{
  "model": "SINet-V2 + BGNet + PVT-v2-B2",
  "architecture": {
    "backbone": "PVT-v2-B2 (pretrained ImageNet)",
    "neck": "RFB_modified x3 (128/320/512 -> 32ch)",
    "bgnet_eam": "EAM: pvt_feats[0](64ch) + pvt_feats[3](512ch) -> edge",
    "bgnet_efm": "EFM x3: edge injection via LocalChannelAttention",
    "decoder": "NCD -> RS5 -> RS4 -> RS3",
    "outputs": "4 mask predictions + 1 edge prediction"
  },
  "hyperparameters": {
    "total_epochs": 70,
    "lr_head": 0.0005,
    "lr_backbone": 5e-05,
    "lr_schedule": "cosine annealing + 3-epoch warmup",
    "batch_size": 32,
    "input_size": 352,
    "optimizer": "Adam",
    "clip_gradient": 0.5,
    "lambda_edge": 3.0,
    "loss_mask": "Weighted BCE + Weighted IoU",
    "loss_edge": "Dice loss (BGNet paper)"
  },
  "datasets": {
    "train": "COD10K-train + CAMO-train (4040 images)",
    "val": "CAMO test set"
  },
  "results": {
    "best_epoch": 37,
    "best_mae_camo": 0.0

## Bước 12 — Zip kết quả để download

In [12]:
import shutil, os

WORK = '/kaggle/working/SINet-V2'
OUT  = '/kaggle/working/SINet_V2_BGNet_Results/'
os.makedirs(OUT + 'checkpoints', exist_ok=True)
os.makedirs(OUT + 'plots',       exist_ok=True)

SNAP = f'{WORK}/snapshot/SINet_V2_BGNet_PVTv2/'
if os.path.exists(SNAP):
    shutil.copytree(SNAP, OUT + 'checkpoints', dirs_exist_ok=True)

if os.path.exists('/kaggle/working/plots_bgnet'):
    shutil.copytree('/kaggle/working/plots_bgnet', OUT + 'plots', dirs_exist_ok=True)

for src in [SNAP + 'training_log.csv',
            '/kaggle/working/training_summary_bgnet.json']:
    if os.path.exists(src):
        shutil.copy(src, OUT)

res_src = f'{WORK}/res/SINet_V2_BGNet_PVTv2/'
if os.path.exists(res_src):
    shutil.copytree(res_src, OUT + 'res', dirs_exist_ok=True)

print('Zipping...')
shutil.make_archive('/kaggle/working/SINet_V2_BGNet_Results', 'zip',
                    '/kaggle/working/SINet_V2_BGNet_Results/')
size = os.path.getsize('/kaggle/working/SINet_V2_BGNet_Results.zip') / (1024**2)
print(f'Done! SINet_V2_BGNet_Results.zip  ({size:.1f} MB)')
print('Go to OUTPUT tab on the right to download.')

Zipping...
Done! SINet_V2_BGNet_Results.zip  (1139.5 MB)
Go to OUTPUT tab on the right to download.
